# School Data Setup - Learner Platform

This notebook creates test data for two schools with:
- Classes 1-10, Divisions A, B, C
- 3 subjects: Math, Science, Social Science
- 2 students per class/division
- 1 teacher per subject per class/division
- Admin users (Tenant Admin, Principal)

## Credential Format
- **Tenant Admin**: `admin@<school>.local` / `password`
- **Principal**: `principal@<school>.local` / `password`
- **Students**: `<class><div><roll>@<school>.local` / `password`
  - Example: `1a01@euroschool.local` / `password`
- **Teachers**: `<class><div>_<subject>_t1@<school>.local` / `password`
  - Example: `1a_math_t1@euroschool.local` / `password`

In [21]:
import requests
import json
from typing import Optional
import time

# Configuration
BASE_URL = "http://localhost:8084"  # API Gateway
DIRECT_URLS = {
    "auth": "http://localhost:8081",
    "tenant": "http://localhost:8082",
    "permission": "http://localhost:8080",
    "profile": "http://localhost:8083"
}

# Use direct URLs for setup (bypass gateway)
USE_DIRECT = True

def get_url(service: str, path: str) -> str:
    if USE_DIRECT:
        return f"{DIRECT_URLS[service]}{path}"
    return f"{BASE_URL}{path}"

# Storage for created IDs (for cleanup)
created_data = {
    "tenants": [],
    "users": [],
    "roles": [],
    "permissions": [],
    "profiles": [],
    "role_assignments": []
}

print("Configuration loaded!")

Configuration loaded!


## Helper Functions

In [22]:
def api_call(method: str, url: str, data: dict = None, headers: dict = None) -> dict:
    """Make API call and return response"""
    default_headers = {"Content-Type": "application/json"}
    if headers:
        default_headers.update(headers)
    
    try:
        if method == "GET":
            resp = requests.get(url, headers=default_headers, timeout=10)
        elif method == "POST":
            resp = requests.post(url, json=data, headers=default_headers, timeout=10)
        elif method == "PUT":
            resp = requests.put(url, json=data, headers=default_headers, timeout=10)
        elif method == "DELETE":
            resp = requests.delete(url, headers=default_headers, timeout=10)
        
        if resp.status_code in [200, 201]:
            return {"success": True, "data": resp.json() if resp.text else {}}
        else:
            return {"success": False, "status": resp.status_code, "error": resp.text}
    except Exception as e:
        return {"success": False, "error": str(e)}

def print_result(operation: str, result: dict):
    """Print operation result"""
    if result["success"]:
        print(f"  [OK] {operation}")
    else:
        print(f"  [FAIL] {operation}: {result.get('error', 'Unknown error')}")

print("Helper functions loaded!")

Helper functions loaded!


## Data Definitions

In [23]:
# Schools configuration
SCHOOLS = [
    {"key": "euroschool", "name": "Euro School"},
    {"key": "ravishankar", "name": "Ravi Shankar Vidyalaya"}
]

# Content Hub - platform-level content creators
CONTENT_HUB = {
    "key": "contenthub",
    "name": "Content Hub",
    "creators": [
        {"email_prefix": "creator1", "name": "Content Creator 1", "subjects": ["math", "science"]},
        {"email_prefix": "creator2", "name": "Content Creator 2", "subjects": ["socialscience"]},
        {"email_prefix": "creator_admin", "name": "Content Hub Admin", "is_admin": True}
    ]
}

# Classes and divisions
CLASSES = list(range(1, 11))  # 1 to 10
DIVISIONS = ["A", "B", "C"]

# Subjects
SUBJECTS = ["math", "science", "socialscience"]
SUBJECT_NAMES = {
    "math": "Mathematics",
    "science": "Science",
    "socialscience": "Social Science"
}

# Students per class/division
STUDENTS_PER_CLASS = 2

# Admin users per school
ADMIN_USERS = [
    {"email_prefix": "admin", "name": "School Administrator", "role": "TENANT_ADMIN"},
    {"email_prefix": "principal", "name": "School Principal", "role": "PRINCIPAL"}
]

# Permission definitions
PERMISSIONS = [
    # Notes permissions
    {"code": "NOTE_CREATE", "resource": "NOTE", "action": "CREATE", "name": "Create Notes", "description": "Create new notes as drafts"},
    {"code": "NOTE_EDIT", "resource": "NOTE", "action": "EDIT", "name": "Edit Notes", "description": "Edit existing notes"},
    {"code": "NOTE_PUBLISH", "resource": "NOTE", "action": "PUBLISH", "name": "Publish Notes", "description": "Publish draft notes"},
    {"code": "NOTE_RELEASE", "resource": "NOTE", "action": "RELEASE", "name": "Release Notes", "description": "Release notes to students"},
    {"code": "NOTE_VIEW", "resource": "NOTE", "action": "VIEW", "name": "View Notes", "description": "View published notes"},
    {"code": "NOTE_READ", "resource": "NOTE", "action": "READ", "name": "Read Notes", "description": "Read note content"},
    {"code": "NOTE_DELETE", "resource": "NOTE", "action": "DELETE", "name": "Delete Notes", "description": "Delete notes"},
    {"code": "NOTE_VIEW_ALL", "resource": "NOTE", "action": "VIEW_ALL", "name": "View All Notes", "description": "View all notes in tenant"},
    {"code": "NOTE_SHARE", "resource": "NOTE", "action": "SHARE", "name": "Share Notes", "description": "Share notes to other tenants"},
    
    # Mindmap permissions
    {"code": "MINDMAP_CREATE", "resource": "MINDMAP", "action": "CREATE", "name": "Create Mindmaps", "description": "Create new mindmaps as drafts"},
    {"code": "MINDMAP_EDIT", "resource": "MINDMAP", "action": "EDIT", "name": "Edit Mindmaps", "description": "Edit existing mindmaps"},
    {"code": "MINDMAP_PUBLISH", "resource": "MINDMAP", "action": "PUBLISH", "name": "Publish Mindmaps", "description": "Publish draft mindmaps"},
    {"code": "MINDMAP_RELEASE", "resource": "MINDMAP", "action": "RELEASE", "name": "Release Mindmaps", "description": "Release mindmaps to students"},
    {"code": "MINDMAP_VIEW", "resource": "MINDMAP", "action": "VIEW", "name": "View Mindmaps", "description": "View published mindmaps"},
    {"code": "MINDMAP_INTERACT", "resource": "MINDMAP", "action": "INTERACT", "name": "Interact with Mindmaps", "description": "Click and interact with mindmaps"},
    {"code": "MINDMAP_DELETE", "resource": "MINDMAP", "action": "DELETE", "name": "Delete Mindmaps", "description": "Delete mindmaps"},
    {"code": "MINDMAP_VIEW_ALL", "resource": "MINDMAP", "action": "VIEW_ALL", "name": "View All Mindmaps", "description": "View all mindmaps in tenant"},
    {"code": "MINDMAP_SHARE", "resource": "MINDMAP", "action": "SHARE", "name": "Share Mindmaps", "description": "Share mindmaps to other tenants"},
    
    # User management permissions
    {"code": "USER_CREATE", "resource": "USER", "action": "CREATE", "name": "Create Users", "description": "Create new users"},
    {"code": "USER_EDIT", "resource": "USER", "action": "EDIT", "name": "Edit Users", "description": "Edit user details"},
    {"code": "USER_DELETE", "resource": "USER", "action": "DELETE", "name": "Delete Users", "description": "Delete users"},
    {"code": "USER_VIEW", "resource": "USER", "action": "VIEW", "name": "View Users", "description": "View user details"},
    {"code": "USER_VIEW_ALL", "resource": "USER", "action": "VIEW_ALL", "name": "View All Users", "description": "View all users in tenant"},
    
    # Role management permissions
    {"code": "ROLE_ASSIGN", "resource": "ROLE", "action": "ASSIGN", "name": "Assign Roles", "description": "Assign roles to users"},
    {"code": "ROLE_REVOKE", "resource": "ROLE", "action": "REVOKE", "name": "Revoke Roles", "description": "Revoke roles from users"},
    
    # Tenant management permissions
    {"code": "TENANT_MANAGE", "resource": "TENANT", "action": "MANAGE", "name": "Manage Tenant", "description": "Manage tenant settings"},
    {"code": "TENANT_VIEW", "resource": "TENANT", "action": "VIEW", "name": "View Tenant", "description": "View tenant details"},
    
    # Content subscription permissions (for schools to access shared content)
    {"code": "CONTENT_SUBSCRIBE", "resource": "CONTENT", "action": "SUBSCRIBE", "name": "Subscribe to Content", "description": "Subscribe to shared content from content hub"},
    {"code": "CONTENT_IMPORT", "resource": "CONTENT", "action": "IMPORT", "name": "Import Content", "description": "Import content from content hub"}
]

# Role definitions with their permissions
ROLE_PERMISSIONS = {
    # Content Hub roles
    "CREATOR": [
        # Create and publish content for sharing
        "NOTE_CREATE", "NOTE_EDIT", "NOTE_PUBLISH", "NOTE_VIEW", "NOTE_READ", "NOTE_SHARE",
        "MINDMAP_CREATE", "MINDMAP_EDIT", "MINDMAP_PUBLISH", "MINDMAP_VIEW", "MINDMAP_INTERACT", "MINDMAP_SHARE"
    ],
    "CONTENT_ADMIN": [
        # Full access to content hub
        "NOTE_CREATE", "NOTE_EDIT", "NOTE_PUBLISH", "NOTE_VIEW", "NOTE_READ", "NOTE_DELETE", "NOTE_VIEW_ALL", "NOTE_SHARE",
        "MINDMAP_CREATE", "MINDMAP_EDIT", "MINDMAP_PUBLISH", "MINDMAP_VIEW", "MINDMAP_INTERACT", "MINDMAP_DELETE", "MINDMAP_VIEW_ALL", "MINDMAP_SHARE",
        "USER_CREATE", "USER_EDIT", "USER_DELETE", "USER_VIEW", "USER_VIEW_ALL",
        "ROLE_ASSIGN", "ROLE_REVOKE",
        "TENANT_MANAGE", "TENANT_VIEW"
    ],
    
    # School roles
    "TENANT_ADMIN": [
        # Full access to everything in school
        "NOTE_CREATE", "NOTE_EDIT", "NOTE_PUBLISH", "NOTE_RELEASE", "NOTE_VIEW", "NOTE_READ", "NOTE_DELETE", "NOTE_VIEW_ALL",
        "MINDMAP_CREATE", "MINDMAP_EDIT", "MINDMAP_PUBLISH", "MINDMAP_RELEASE", "MINDMAP_VIEW", "MINDMAP_INTERACT", "MINDMAP_DELETE", "MINDMAP_VIEW_ALL",
        "USER_CREATE", "USER_EDIT", "USER_DELETE", "USER_VIEW", "USER_VIEW_ALL",
        "ROLE_ASSIGN", "ROLE_REVOKE",
        "TENANT_MANAGE", "TENANT_VIEW",
        "CONTENT_SUBSCRIBE", "CONTENT_IMPORT"
    ],
    "PRINCIPAL": [
        # Can view all content but not modify
        "NOTE_VIEW", "NOTE_READ", "NOTE_VIEW_ALL",
        "MINDMAP_VIEW", "MINDMAP_INTERACT", "MINDMAP_VIEW_ALL",
        "USER_VIEW", "USER_VIEW_ALL",
        "TENANT_VIEW"
    ],
    "TEACHER": [
        "NOTE_CREATE", "NOTE_EDIT", "NOTE_PUBLISH", "NOTE_RELEASE", "NOTE_VIEW", "NOTE_READ",
        "MINDMAP_CREATE", "MINDMAP_EDIT", "MINDMAP_PUBLISH", "MINDMAP_RELEASE", "MINDMAP_VIEW", "MINDMAP_INTERACT"
    ],
    "STUDENT": [
        "NOTE_VIEW", "NOTE_READ",
        "MINDMAP_VIEW", "MINDMAP_INTERACT"
    ]
}

print(f"Data definitions loaded!")
print(f"  Schools: {len(SCHOOLS)}")
print(f"  Content Hub: {CONTENT_HUB['name']} ({len(CONTENT_HUB['creators'])} creators)")
print(f"  Classes: {len(CLASSES)}")
print(f"  Divisions: {len(DIVISIONS)}")
print(f"  Subjects: {len(SUBJECTS)}")
print(f"  Admin users per school: {len(ADMIN_USERS)}")
print(f"  Total students per school: {len(CLASSES) * len(DIVISIONS) * STUDENTS_PER_CLASS}")
print(f"  Total teachers per school: {len(CLASSES) * len(DIVISIONS) * len(SUBJECTS)}")

Data definitions loaded!
  Schools: 2
  Content Hub: Content Hub (3 creators)
  Classes: 10
  Divisions: 3
  Subjects: 3
  Admin users per school: 2
  Total students per school: 60
  Total teachers per school: 90


---
# CREATE SECTION
---

## Step 1: Create Tenants (Schools)

In [24]:
print("Creating tenants (schools + content hub)...\n")

tenant_ids = {}

def get_or_create_tenant(tenant_key: str, tenant_name: str, policy: dict, tenant_type: str) -> str:
    """Get existing tenant or create new one. Returns tenant ID."""
    # First, try to resolve existing tenant
    resolve_url = get_url("tenant", f"/v1/resolve?tenantKey={tenant_key}")
    result = api_call("GET", resolve_url)
    
    if result["success"]:
        # Resolve endpoint returns 'tenantId' not 'id'
        tenant_id = result["data"].get("tenantId") or result["data"].get("id")
        print(f"  [EXISTS] {tenant_type}: {tenant_name}")
        print(f"      ID: {tenant_id}")
        return tenant_id
    
    # Tenant doesn't exist, create it
    create_url = get_url("tenant", "/v1/tenants")
    data = {
        "tenantKey": tenant_key,
        "name": tenant_name,
        "policy": policy
    }
    
    result = api_call("POST", create_url, data)
    
    if result["success"]:
        tenant_id = result["data"]["id"]
        print(f"  [OK] {tenant_type}: {tenant_name}")
        print(f"      ID: {tenant_id}")
        created_data["tenants"].append({"id": tenant_id, "key": tenant_key, "type": tenant_type.lower()})
        return tenant_id
    else:
        print(f"  [FAIL] {tenant_type}: {tenant_name}: {result.get('error', '')[:50]}")
        return None

# Create/Get School Tenants
for school in SCHOOLS:
    tenant_id = get_or_create_tenant(
        tenant_key=school["key"],
        tenant_name=school["name"],
        policy={
            "allowedAuthMethods": ["PASSWORD"],
            "selfSignupEnabled": False,
            "approvalRequired": True
        },
        tenant_type="School"
    )
    if tenant_id:
        tenant_ids[school["key"]] = tenant_id

# Create/Get Content Hub Tenant
tenant_id = get_or_create_tenant(
    tenant_key=CONTENT_HUB["key"],
    tenant_name=CONTENT_HUB["name"],
    policy={
        "allowedAuthMethods": ["PASSWORD"],
        "selfSignupEnabled": False,
        "approvalRequired": False
    },
    tenant_type="Content Hub"
)
if tenant_id:
    tenant_ids[CONTENT_HUB["key"]] = tenant_id

print(f"\nTotal tenants: {len(tenant_ids)} (2 schools + 1 content hub)")

Creating tenants (schools + content hub)...

  [EXISTS] School: Euro School
      ID: d86f5455-3701-431a-ada6-7f3c5e3c38d9
  [EXISTS] School: Ravi Shankar Vidyalaya
      ID: a0ba4b2e-d244-4f73-b54a-60dc22ee7f15
  [EXISTS] Content Hub: Content Hub
      ID: d4ca53ee-ce9a-40c7-a932-73aaf72714ae

Total tenants: 3 (2 schools + 1 content hub)


## Step 2: Create Permissions (Global)

In [25]:
print("Creating permissions...\n")

permission_ids = {}

for perm in PERMISSIONS:
    perm_code = perm["code"]
    
    # First, check if permission already exists
    get_url_path = get_url("permission", f"/permissions/{perm_code}")
    result = api_call("GET", get_url_path)
    
    if result["success"]:
        # Permission already exists
        perm_id = result["data"]["id"]
        actual_code = result["data"].get("code", perm_code)
        permission_ids[perm_code] = {"id": perm_id, "code": actual_code}
        print(f"  [EXISTS] Permission: {perm_code}")
        continue
    
    # Permission doesn't exist, create it
    url = get_url("permission", "/permissions")
    data = {
        "name": perm_code,  # Use code (e.g., "NOTE_CREATE") as name so it becomes the permission code
        "description": perm["description"],
        "resource": perm["resource"],
        "action": perm["action"],
        "active": True
    }
    
    result = api_call("POST", url, data)
    
    if result["success"]:
        perm_id = result["data"]["id"]
        actual_code = result["data"].get("code", perm_code)
        permission_ids[perm_code] = {"id": perm_id, "code": actual_code}
        created_data["permissions"].append({"id": perm_id, "code": perm_code})
        print(f"  [OK] Permission: {perm_code}")
    else:
        print(f"  [FAIL] Permission: {perm_code}: {result.get('error', '')[:50]}")

print(f"\nTotal permissions: {len(permission_ids)}")

Creating permissions...

  [OK] Permission: NOTE_CREATE
  [OK] Permission: NOTE_EDIT
  [OK] Permission: NOTE_PUBLISH
  [OK] Permission: NOTE_RELEASE
  [OK] Permission: NOTE_VIEW
  [OK] Permission: NOTE_READ
  [OK] Permission: NOTE_DELETE
  [OK] Permission: NOTE_VIEW_ALL
  [OK] Permission: NOTE_SHARE
  [OK] Permission: MINDMAP_CREATE
  [OK] Permission: MINDMAP_EDIT
  [OK] Permission: MINDMAP_PUBLISH
  [OK] Permission: MINDMAP_RELEASE
  [OK] Permission: MINDMAP_VIEW
  [OK] Permission: MINDMAP_INTERACT
  [OK] Permission: MINDMAP_DELETE
  [OK] Permission: MINDMAP_VIEW_ALL
  [OK] Permission: MINDMAP_SHARE
  [OK] Permission: USER_CREATE
  [OK] Permission: USER_EDIT
  [OK] Permission: USER_DELETE
  [OK] Permission: USER_VIEW
  [OK] Permission: USER_VIEW_ALL
  [OK] Permission: ROLE_ASSIGN
  [OK] Permission: ROLE_REVOKE
  [OK] Permission: TENANT_MANAGE
  [OK] Permission: TENANT_VIEW
  [OK] Permission: CONTENT_SUBSCRIBE
  [OK] Permission: CONTENT_IMPORT

Total permissions: 29


## Step 3: Create Roles per Tenant

In [26]:
print("Creating roles for each tenant...\n")

role_ids = {}  # {tenant_key: {role_name: role_id}}

# Define which roles belong to which tenant type
SCHOOL_ROLES = ["TENANT_ADMIN", "PRINCIPAL", "TEACHER", "STUDENT"]
CONTENT_HUB_ROLES = ["CONTENT_ADMIN", "CREATOR"]

for tenant_key, tenant_id in tenant_ids.items():
    print(f"\n--- {tenant_key} ---")
    role_ids[tenant_key] = {}
    
    # Get existing roles for this tenant
    get_roles_url = get_url("permission", f"/tenants/{tenant_id}/roles")
    result = api_call("GET", get_roles_url)
    existing_roles = {}
    if result["success"]:
        for role in result["data"]:
            existing_roles[role["name"]] = role["id"]
    
    # Determine which roles to create based on tenant type
    if tenant_key == CONTENT_HUB["key"]:
        roles_to_create = CONTENT_HUB_ROLES
    else:
        roles_to_create = SCHOOL_ROLES
    
    for role_name in roles_to_create:
        if role_name not in ROLE_PERMISSIONS:
            print(f"  [SKIP] Role {role_name} not defined in ROLE_PERMISSIONS")
            continue
        
        # Check if role already exists
        if role_name in existing_roles:
            role_id = existing_roles[role_name]
            role_ids[tenant_key][role_name] = role_id
            print(f"  [EXISTS] Role: {role_name}")
            continue
            
        url = get_url("permission", f"/tenants/{tenant_id}/roles")
        data = {
            "name": role_name,
            "description": f"{role_name} role for {tenant_key}",
            "active": True
        }
        
        result = api_call("POST", url, data)
        
        if result["success"]:
            role_id = result["data"]["id"]
            role_ids[tenant_key][role_name] = role_id
            created_data["roles"].append({"tenant_id": tenant_id, "role_id": role_id, "name": role_name})
            print(f"  [OK] Role: {role_name}")
        else:
            print(f"  [FAIL] Role: {role_name}: {result.get('error', '')[:50]}")

print(f"\nCreated roles for {len(role_ids)} tenants")

Creating roles for each tenant...


--- euroschool ---
  [OK] Role: TENANT_ADMIN
  [OK] Role: PRINCIPAL
  [OK] Role: TEACHER
  [OK] Role: STUDENT

--- ravishankar ---
  [OK] Role: TENANT_ADMIN
  [OK] Role: PRINCIPAL
  [OK] Role: TEACHER
  [OK] Role: STUDENT

--- contenthub ---
  [OK] Role: CONTENT_ADMIN
  [OK] Role: CREATOR

Created roles for 3 tenants


## Step 4: Assign Permissions to Roles

In [27]:
print("Assigning permissions to roles...\n")

for tenant_key, tenant_id in tenant_ids.items():
    print(f"\n--- {tenant_key} ---")
    
    for role_name, perm_codes in ROLE_PERMISSIONS.items():
        role_id = role_ids.get(tenant_key, {}).get(role_name)
        if not role_id:
            print(f"  [SKIP] Role {role_name} not found")
            continue
        
        print(f"  Role: {role_name}")
        
        # Get existing grants for this role
        get_grants_url = get_url("permission", f"/tenants/{tenant_id}/roles/{role_id}/grants")
        result = api_call("GET", get_grants_url)
        existing_grants = set()
        if result["success"]:
            for grant in result["data"]:
                existing_grants.add(grant["permissionCode"])
        
        for perm_code in perm_codes:
            perm_info = permission_ids.get(perm_code)
            if not perm_info:
                print(f"    [SKIP] Permission {perm_code} not found")
                continue
            
            # Check if grant already exists
            if perm_info["code"] in existing_grants:
                print(f"    [EXISTS] {perm_code}")
                continue
            
            url = get_url("permission", f"/tenants/{tenant_id}/roles/{role_id}/grants")
            data = {
                "permissionCode": perm_info["code"],
                "scopeCode": "OWN" if role_name == "TEACHER" else "CLASS"
            }
            
            result = api_call("POST", url, data)
            if result["success"]:
                print(f"    [OK] {perm_code}")
            else:
                print(f"    [FAIL] {perm_code}: {result.get('error', '')[:50]}")

print("\nPermission grants completed!")

Assigning permissions to roles...


--- euroschool ---
  [SKIP] Role CREATOR not found
  [SKIP] Role CONTENT_ADMIN not found
  Role: TENANT_ADMIN
    [OK] NOTE_CREATE
    [OK] NOTE_EDIT
    [OK] NOTE_PUBLISH
    [OK] NOTE_RELEASE
    [OK] NOTE_VIEW
    [OK] NOTE_READ
    [OK] NOTE_DELETE
    [OK] NOTE_VIEW_ALL
    [OK] MINDMAP_CREATE
    [OK] MINDMAP_EDIT
    [OK] MINDMAP_PUBLISH
    [OK] MINDMAP_RELEASE
    [OK] MINDMAP_VIEW
    [OK] MINDMAP_INTERACT
    [OK] MINDMAP_DELETE
    [OK] MINDMAP_VIEW_ALL
    [OK] USER_CREATE
    [OK] USER_EDIT
    [OK] USER_DELETE
    [OK] USER_VIEW
    [OK] USER_VIEW_ALL
    [OK] ROLE_ASSIGN
    [OK] ROLE_REVOKE
    [OK] TENANT_MANAGE
    [OK] TENANT_VIEW
    [OK] CONTENT_SUBSCRIBE
    [OK] CONTENT_IMPORT
  Role: PRINCIPAL
    [OK] NOTE_VIEW
    [OK] NOTE_READ
    [OK] NOTE_VIEW_ALL
    [OK] MINDMAP_VIEW
    [OK] MINDMAP_INTERACT
    [OK] MINDMAP_VIEW_ALL
    [OK] USER_VIEW
    [OK] USER_VIEW_ALL
    [OK] TENANT_VIEW
  Role: TEACHER
    [OK] NOTE_CREATE
 

## Step 5: Create Users (Students and Teachers)

In [28]:
print("Creating users...\n")

user_ids = {}  # {tenant_key: {email: {user_id, role, ...}}}
DEFAULT_PASSWORD = "password"

def get_or_create_user(tenant_id: str, email: str, password: str, name: str) -> str:
    """Get existing user via login or create new one. Returns user ID."""
    # First, try to login to check if user exists
    login_url = get_url("auth", "/auth/login")
    login_data = {
        "tenantId": tenant_id,
        "identifier": email,
        "password": password
    }
    result = api_call("POST", login_url, login_data)
    
    # Check if login returned a valid existing user (not a mock random ID)
    # The mock service returns a userId even if user doesn't exist, 
    # so we'll try signup and handle errors
    
    # Try signup
    signup_url = get_url("auth", "/auth/signup")
    signup_data = {
        "tenantId": tenant_id,
        "email": email,
        "password": password,
        "name": name,
        "joinMethod": "SIGNUP"
    }
    
    result = api_call("POST", signup_url, signup_data)
    
    if result["success"]:
        return result["data"].get("userId"), True  # new user
    
    # If signup failed, try to get user ID from login response
    login_result = api_call("POST", login_url, login_data)
    if login_result["success"] and login_result["data"].get("userId"):
        return login_result["data"].get("userId"), False  # existing user
    
    return None, False

# ============================================================
# Create Content Hub Users (Creators)
# ============================================================
contenthub_key = CONTENT_HUB["key"]
contenthub_tenant_id = tenant_ids.get(contenthub_key)

if contenthub_tenant_id:
    print(f"\n{'='*50}")
    print(f"Content Hub: {CONTENT_HUB['name']}")
    print(f"{'='*50}")
    
    user_ids[contenthub_key] = {}
    creator_count = 0
    existing_count = 0
    
    for creator in CONTENT_HUB["creators"]:
        email = f"{creator['email_prefix']}@{contenthub_key}.local"
        name = creator["name"]
        role = "CONTENT_ADMIN" if creator.get("is_admin") else "CREATOR"
        
        user_id, is_new = get_or_create_user(contenthub_tenant_id, email, DEFAULT_PASSWORD, name)
        
        if user_id:
            user_ids[contenthub_key][email] = {
                "user_id": user_id,
                "role": role,
                "name": name,
                "subjects": creator.get("subjects", []),
                "is_admin": creator.get("is_admin", False)
            }
            if is_new:
                created_data["users"].append({"tenant_id": contenthub_tenant_id, "user_id": user_id, "email": email})
                creator_count += 1
                print(f"  [OK] {role}: {email}")
            else:
                existing_count += 1
                print(f"  [EXISTS] {role}: {email}")
        else:
            print(f"  [FAIL] {email}")
    
    print(f"\n  Created {creator_count} new, {existing_count} existing content hub users")

# ============================================================
# Create School Users (Admins, Teachers, Students)
# ============================================================
for school in SCHOOLS:
    school_key = school["key"]
    tenant_id = tenant_ids.get(school_key)
    
    if not tenant_id:
        print(f"[SKIP] Tenant not found for {school_key}")
        continue
    
    print(f"\n{'='*50}")
    print(f"School: {school['name']}")
    print(f"{'='*50}")
    
    user_ids[school_key] = {}
    admin_count = 0
    student_count = 0
    teacher_count = 0
    existing_count = 0
    
    # Create Admin Users (TENANT_ADMIN, PRINCIPAL)
    print("\n  Creating admin users...")
    for admin in ADMIN_USERS:
        email = f"{admin['email_prefix']}@{school_key}.local"
        name = f"{admin['name']} - {school['name']}"
        
        user_id, is_new = get_or_create_user(tenant_id, email, DEFAULT_PASSWORD, name)
        
        if user_id:
            user_ids[school_key][email] = {
                "user_id": user_id,
                "role": admin["role"],
                "class": None,
                "division": None,
                "name": name,
                "is_admin": True
            }
            if is_new:
                created_data["users"].append({"tenant_id": tenant_id, "user_id": user_id, "email": email})
                admin_count += 1
                print(f"    [OK] {admin['role']}: {email}")
            else:
                existing_count += 1
                print(f"    [EXISTS] {admin['role']}: {email}")
        else:
            print(f"    [FAIL] {admin['role']} {email}")
    
    # Create Students and Teachers
    print("\n  Creating students and teachers...")
    for class_num in CLASSES:
        for division in DIVISIONS:
            # Create Students
            for roll in range(1, STUDENTS_PER_CLASS + 1):
                email = f"{class_num}{division.lower()}{roll:02d}@{school_key}.local"
                name = f"Student {class_num}{division}{roll:02d}"
                
                user_id, is_new = get_or_create_user(tenant_id, email, DEFAULT_PASSWORD, name)
                
                if user_id:
                    user_ids[school_key][email] = {
                        "user_id": user_id,
                        "role": "STUDENT",
                        "class": class_num,
                        "division": division,
                        "name": name
                    }
                    if is_new:
                        created_data["users"].append({"tenant_id": tenant_id, "user_id": user_id, "email": email})
                        student_count += 1
                    else:
                        existing_count += 1
            
            # Create Teachers (one per subject)
            for subject in SUBJECTS:
                email = f"{class_num}{division.lower()}_{subject}_t1@{school_key}.local"
                name = f"Teacher {SUBJECT_NAMES[subject]} {class_num}{division}"
                
                user_id, is_new = get_or_create_user(tenant_id, email, DEFAULT_PASSWORD, name)
                
                if user_id:
                    user_ids[school_key][email] = {
                        "user_id": user_id,
                        "role": "TEACHER",
                        "class": class_num,
                        "division": division,
                        "subject": subject,
                        "name": name
                    }
                    if is_new:
                        created_data["users"].append({"tenant_id": tenant_id, "user_id": user_id, "email": email})
                        teacher_count += 1
                    else:
                        existing_count += 1
    
    print(f"\n  Summary: {admin_count} admins, {student_count} students, {teacher_count} teachers (new)")
    print(f"           {existing_count} existing users")

print(f"\n\nTotal new users created: {len(created_data['users'])}")

Creating users...


Content Hub: Content Hub
  [EXISTS] CREATOR: creator1@contenthub.local
  [EXISTS] CREATOR: creator2@contenthub.local
  [EXISTS] CONTENT_ADMIN: creator_admin@contenthub.local

  Created 0 new, 3 existing content hub users

School: Euro School

  Creating admin users...
    [EXISTS] TENANT_ADMIN: admin@euroschool.local
    [EXISTS] PRINCIPAL: principal@euroschool.local

  Creating students and teachers...

  Summary: 0 admins, 0 students, 0 teachers (new)
           152 existing users

School: Ravi Shankar Vidyalaya

  Creating admin users...
    [EXISTS] TENANT_ADMIN: admin@ravishankar.local
    [EXISTS] PRINCIPAL: principal@ravishankar.local

  Creating students and teachers...

  Summary: 0 admins, 0 students, 0 teachers (new)
           152 existing users


Total new users created: 0


## Step 6: Assign Roles to Users

In [29]:
print("Assigning roles to users...\n")

assignment_count = 0
existing_count = 0

for tenant_key, users in user_ids.items():
    tenant_id = tenant_ids.get(tenant_key)
    if not tenant_id:
        continue
    
    print(f"\n--- {tenant_key} ---")
    
    for email, user_info in users.items():
        user_id = user_info["user_id"]
        role_name = user_info["role"]
        
        role_id = role_ids.get(tenant_key, {}).get(role_name)
        if not role_id:
            print(f"  [SKIP] Role {role_name} not found for {email}")
            continue
        
        # Get existing role assignments for this user
        get_assignments_url = get_url("permission", f"/tenants/{tenant_id}/users/{user_id}/roles")
        result = api_call("GET", get_assignments_url)
        existing_role_ids = set()
        if result["success"]:
            for assignment in result["data"]:
                existing_role_ids.add(assignment["roleId"])
        
        # Check if role is already assigned
        if role_id in existing_role_ids:
            existing_count += 1
            if user_info.get("is_admin") or role_name in ["CREATOR", "CONTENT_ADMIN"]:
                print(f"  [EXISTS] {role_name}: {email}")
            continue
        
        # Determine scope based on role and tenant type
        if tenant_key == CONTENT_HUB["key"]:
            scope_type = "TENANT"
            scope_id = tenant_key
        elif user_info.get("is_admin") or role_name in ["TENANT_ADMIN", "PRINCIPAL"]:
            scope_type = "TENANT"
            scope_id = tenant_key
        else:
            class_num = user_info.get("class")
            division = user_info.get("division")
            scope_type = "CLASS_DIVISION"
            scope_id = f"{class_num}{division}"
        
        url = get_url("permission", f"/tenants/{tenant_id}/users/{user_id}/roles")
        data = {
            "roleId": role_id,
            "scopeType": scope_type,
            "scopeId": scope_id,
            "status": "ACTIVE"
        }
        
        result = api_call("POST", url, data)
        
        if result["success"]:
            assignment_id = result["data"].get("id")
            created_data["role_assignments"].append({
                "tenant_id": tenant_id,
                "user_id": user_id,
                "assignment_id": assignment_id
            })
            assignment_count += 1
            if user_info.get("is_admin") or role_name in ["CREATOR", "CONTENT_ADMIN"]:
                print(f"  [OK] {role_name}: {email}")
        else:
            print(f"  [FAIL] {email}: {result.get('error', '')[:50]}")
    
    print(f"  Processed {len(users)} users")

print(f"\nNew role assignments: {assignment_count}, existing: {existing_count}")

Assigning roles to users...


--- contenthub ---
  [OK] CREATOR: creator1@contenthub.local
  [OK] CREATOR: creator2@contenthub.local
  [OK] CONTENT_ADMIN: creator_admin@contenthub.local
  Processed 3 users

--- euroschool ---
  [OK] TENANT_ADMIN: admin@euroschool.local
  [OK] PRINCIPAL: principal@euroschool.local
  Processed 152 users

--- ravishankar ---
  [OK] TENANT_ADMIN: admin@ravishankar.local
  [OK] PRINCIPAL: principal@ravishankar.local
  Processed 152 users

New role assignments: 307, existing: 0


## Step 7: Create User Profiles

In [30]:
print("Creating user profiles...\n")

profile_count = 0
existing_count = 0

for school_key, users in user_ids.items():
    tenant_id = tenant_ids.get(school_key)
    if not tenant_id:
        continue
    
    print(f"\n--- {school_key} ---")
    
    for email, user_info in users.items():
        user_id = user_info["user_id"]
        user_type = user_info["role"]
        
        # Check if profile already exists
        get_profile_url = get_url("profile", f"/v1/tenants/{tenant_id}/profiles/by-user/{user_id}")
        result = api_call("GET", get_profile_url)
        
        if result["success"]:
            existing_count += 1
            continue
        
        # Profile doesn't exist, create it
        url = get_url("profile", f"/v1/tenants/{tenant_id}/profiles")
        data = {
            "userId": user_id,
            "displayName": user_info["name"],
            "email": email,
            "userType": user_type,
            "status": "ACTIVE"
        }
        
        result = api_call("POST", url, data)
        
        if result["success"]:
            profile_id = result["data"].get("id")
            created_data["profiles"].append({
                "tenant_id": tenant_id,
                "profile_id": profile_id
            })
            profile_count += 1
    
    print(f"  Created {profile_count} new profiles, {existing_count} existing")

print(f"\nTotal new profiles: {profile_count}, existing: {existing_count}")

Creating user profiles...


--- contenthub ---
  Created 0 new profiles, 0 existing

--- euroschool ---
  Created 151 new profiles, 0 existing

--- ravishankar ---
  Created 302 new profiles, 0 existing

Total new profiles: 302, existing: 0


## Summary - Created Data

In [31]:
print("="*60)
print("CREATION SUMMARY")
print("="*60)
print(f"\nTenants created: {len(created_data['tenants'])}")
print(f"Permissions created: {len(created_data['permissions'])}")
print(f"Roles created: {len(created_data['roles'])}")
print(f"Users created: {len(created_data['users'])}")
print(f"Role assignments: {len(created_data['role_assignments'])}")
print(f"Profiles created: {len(created_data['profiles'])}")

print("\n" + "="*60)
print("TENANT IDs")
print("="*60)
for tenant_key, tenant_id in tenant_ids.items():
    tenant_type = "Content Hub" if tenant_key == "contenthub" else "School"
    print(f"  {tenant_key} ({tenant_type}): {tenant_id}")

print("\n" + "="*60)
print("SAMPLE CREDENTIALS (password: 'password' for all)")
print("="*60)

print("\n--- CONTENT HUB (creates content for all schools) ---")
print("Content Admin:")
print("  creator_admin@contenthub.local")
print("\nContent Creators:")
print("  creator1@contenthub.local (Math, Science)")
print("  creator2@contenthub.local (Social Science)")

print("\n--- SCHOOL ADMINS ---")
print("Tenant Admin (full access):")
print("  admin@euroschool.local")
print("  admin@ravishankar.local")
print("\nPrincipal (view all):")
print("  principal@euroschool.local")
print("  principal@ravishankar.local")

print("\n--- TEACHERS ---")
print("Format: <class><div>_<subject>_t1@<school>.local")
print("  1a_math_t1@euroschool.local")
print("  1a_science_t1@euroschool.local")
print("  5b_socialscience_t1@ravishankar.local")

print("\n--- STUDENTS ---")
print("Format: <class><div><roll>@<school>.local")
print("  1a01@euroschool.local")
print("  1a02@euroschool.local")
print("  5b01@ravishankar.local")

print("\n" + "="*60)
print("ROLE HIERARCHY")
print("="*60)
print("""
CONTENT HUB ROLES:
  CONTENT_ADMIN
    - Full access to content hub
    - Manage creators, content, settings
    - Scope: TENANT (contenthub)
    
  CREATOR
    - Create and publish content
    - Share content to schools
    - Scope: TENANT (contenthub)

SCHOOL ROLES:
  TENANT_ADMIN (School Admin)
    - Full access to all notes, mindmaps, users
    - Can import content from content hub
    - Scope: TENANT (school-wide)
    
  PRINCIPAL
    - Read-only access to all content
    - Scope: TENANT (school-wide)
    
  TEACHER
    - Create, edit, publish, release own content
    - Cannot see other teachers' content
    - Scope: CLASS_DIVISION (e.g., 1A, 2B)
    
  STUDENT  
    - View and read shared content only
    - Scope: CLASS_DIVISION (e.g., 1A, 2B)
""")

CREATION SUMMARY

Tenants created: 0
Permissions created: 29
Roles created: 10
Users created: 0
Role assignments: 307
Profiles created: 302

TENANT IDs
  euroschool (School): d86f5455-3701-431a-ada6-7f3c5e3c38d9
  ravishankar (School): a0ba4b2e-d244-4f73-b54a-60dc22ee7f15
  contenthub (Content Hub): d4ca53ee-ce9a-40c7-a932-73aaf72714ae

SAMPLE CREDENTIALS (password: 'password' for all)

--- CONTENT HUB (creates content for all schools) ---
Content Admin:
  creator_admin@contenthub.local

Content Creators:
  creator1@contenthub.local (Math, Science)
  creator2@contenthub.local (Social Science)

--- SCHOOL ADMINS ---
Tenant Admin (full access):
  admin@euroschool.local
  admin@ravishankar.local

Principal (view all):
  principal@euroschool.local
  principal@ravishankar.local

--- TEACHERS ---
Format: <class><div>_<subject>_t1@<school>.local
  1a_math_t1@euroschool.local
  1a_science_t1@euroschool.local
  5b_socialscience_t1@ravishankar.local

--- STUDENTS ---
Format: <class><div><roll>@<

## Save Created Data to File

In [32]:
# Save created data for later cleanup
import json
from datetime import datetime

output_data = {
    "created_at": datetime.now().isoformat(),
    "tenant_ids": tenant_ids,
    "role_ids": role_ids,
    "permission_ids": permission_ids,
    "user_ids": user_ids,
    "created_data": created_data
}

with open("school_data_created.json", "w") as f:
    json.dump(output_data, f, indent=2, default=str)

print("Data saved to school_data_created.json")

Data saved to school_data_created.json


---
# DELETE SECTION
---

Run the cells below to delete all created data. 

**WARNING**: This will permanently delete all users, roles, and tenants created above!

## Load Created Data (if needed)

In [ ]:
# Load previously created data from file
import json
import os

if os.path.exists("school_data_created.json"):
    with open("school_data_created.json", "r") as f:
        loaded_data = json.load(f)
    
    tenant_ids = loaded_data.get("tenant_ids", {})
    role_ids = loaded_data.get("role_ids", {})
    permission_ids = loaded_data.get("permission_ids", {})
    user_ids = loaded_data.get("user_ids", {})
    created_data = loaded_data.get("created_data", {})
    
    print("Loaded data from school_data_created.json")
    print(f"  Tenants: {len(tenant_ids)}")
    print(f"  Users: {len(created_data.get('users', []))}")
else:
    print("No saved data found. Run the CREATE section first.")

## Delete Users

In [ ]:
print("Deleting users...\n")

deleted_users = 0
failed_users = 0

for user in created_data.get("users", []):
    user_id = user.get("user_id")
    if not user_id:
        continue
    
    url = get_url("auth", f"/auth/users/{user_id}")
    result = api_call("DELETE", url)
    
    if result["success"]:
        deleted_users += 1
    else:
        failed_users += 1

print(f"Deleted {deleted_users} users, {failed_users} failed")

## Delete Roles

In [ ]:
print("Deleting roles...\n")

deleted_roles = 0
failed_roles = 0

for role in created_data.get("roles", []):
    tenant_id = role.get("tenant_id")
    role_id = role.get("role_id")
    if not tenant_id or not role_id:
        continue
    
    url = get_url("permission", f"/tenants/{tenant_id}/roles/{role_id}")
    result = api_call("DELETE", url)
    
    if result["success"]:
        deleted_roles += 1
    else:
        failed_roles += 1

print(f"Deleted {deleted_roles} roles, {failed_roles} failed")

## Delete Tenants

In [ ]:
print("Deleting tenants...\n")

deleted_tenants = 0
failed_tenants = 0

for tenant in created_data.get("tenants", []):
    tenant_id = tenant.get("id")
    if not tenant_id:
        continue
    
    url = get_url("tenant", f"/v1/tenants/{tenant_id}")
    result = api_call("DELETE", url)
    
    if result["success"]:
        deleted_tenants += 1
        print(f"  [OK] Deleted tenant: {tenant.get('key')}")
    else:
        failed_tenants += 1
        print(f"  [FAIL] Tenant {tenant.get('key')}: {result.get('error', '')[:50]}")

print(f"\nDeleted {deleted_tenants} tenants, {failed_tenants} failed")

## Cleanup JSON File

In [ ]:
import os

if os.path.exists("school_data_created.json"):
    os.remove("school_data_created.json")
    print("Removed school_data_created.json")
else:
    print("No file to remove")

# Clear in-memory data
tenant_ids = {}
role_ids = {}
permission_ids = {}
user_ids = {}
created_data = {
    "tenants": [],
    "users": [],
    "roles": [],
    "permissions": [],
    "profiles": [],
    "role_assignments": []
}

print("In-memory data cleared")
print("\nDeletion complete!")

---
# VERIFICATION SECTION
---

## Test Login

In [ ]:
# Test login with sample credentials

def test_login(tenant_id: str, email: str, password: str = "password"):
    """Test login and return result"""
    url = get_url("auth", "/auth/login")
    data = {
        "tenantId": tenant_id,
        "identifier": email,
        "password": password
    }
    result = api_call("POST", url, data)
    return result

# Test Content Hub logins
if tenant_ids.get("contenthub"):
    print("="*60)
    print("TESTING LOGIN - CONTENT HUB")
    print("="*60)
    
    print("\n1. Testing CONTENT_ADMIN: creator_admin@contenthub.local")
    result = test_login(tenant_ids["contenthub"], "creator_admin@contenthub.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")
    
    print("\n2. Testing CREATOR: creator1@contenthub.local")
    result = test_login(tenant_ids["contenthub"], "creator1@contenthub.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")

# Test School logins
if tenant_ids.get("euroschool"):
    print("\n" + "="*60)
    print("TESTING LOGIN - EUROSCHOOL")
    print("="*60)
    
    print("\n1. Testing TENANT_ADMIN: admin@euroschool.local")
    result = test_login(tenant_ids["euroschool"], "admin@euroschool.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")
    
    print("\n2. Testing PRINCIPAL: principal@euroschool.local")
    result = test_login(tenant_ids["euroschool"], "principal@euroschool.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")
    
    print("\n3. Testing TEACHER: 1a_math_t1@euroschool.local")
    result = test_login(tenant_ids["euroschool"], "1a_math_t1@euroschool.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")
    
    print("\n4. Testing STUDENT: 1a01@euroschool.local")
    result = test_login(tenant_ids["euroschool"], "1a01@euroschool.local")
    if result["success"]:
        print(f"   [OK] Login successful! User ID: {result['data'].get('userId')}")
    else:
        print(f"   [FAIL] {result.get('error', '')[:80]}")
else:
    print("Run the CREATE section first to test login")

## List All Users (Content Hub + Schools)

In [ ]:
# List all created users

for tenant_key, users in user_ids.items():
    print(f"\n{'='*60}")
    
    if tenant_key == "contenthub":
        print(f"CONTENT HUB")
        print(f"{'='*60}")
        
        admins = [(email, info) for email, info in users.items() if info.get("is_admin")]
        creators = [(email, info) for email, info in users.items() if info["role"] == "CREATOR"]
        
        print(f"\nContent Admins ({len(admins)}):")
        for email, info in sorted(admins):
            print(f"  {email} - {info['role']}")
        
        print(f"\nContent Creators ({len(creators)}):")
        for email, info in sorted(creators):
            subjects = ", ".join(info.get("subjects", []))
            print(f"  {email} - subjects: {subjects}")
        
        print(f"\n  Total: {len(users)} content hub users")
    else:
        print(f"School: {tenant_key.upper()}")
        print(f"{'='*60}")
        
        admins = [(email, info) for email, info in users.items() if info.get("is_admin")]
        students = [(email, info) for email, info in users.items() if info["role"] == "STUDENT"]
        teachers = [(email, info) for email, info in users.items() if info["role"] == "TEACHER"]
        
        print(f"\nAdmins ({len(admins)}):")
        for email, info in sorted(admins):
            print(f"  {email} - {info['role']}")
        
        print(f"\nStudents ({len(students)}):")
        for email, info in sorted(students)[:10]:  # Show first 10
            print(f"  {email} - Class {info['class']}{info['division']}")
        if len(students) > 10:
            print(f"  ... and {len(students) - 10} more")
        
        print(f"\nTeachers ({len(teachers)}):")
        for email, info in sorted(teachers)[:10]:  # Show first 10
            print(f"  {email} - {info.get('subject', 'N/A')}")
        if len(teachers) > 10:
            print(f"  ... and {len(teachers) - 10} more")
        
        print(f"\n  Total: {len(users)} users")

---
# CONTENT CREATION SECTION
---

Create sample classes, notes, and mindmaps for teachers

## Step 8: Add Content Service URLs

In [ ]:
# Add content service URLs
DIRECT_URLS["notes"] = "http://localhost:8088"
DIRECT_URLS["mindmap"] = "http://localhost:8087"
DIRECT_URLS["workflow"] = "http://localhost:8086"

# Sample content templates
SAMPLE_NOTES = {
    "math": [
        {"title": "Introduction to Algebra", "content": "# Algebra Basics\n\nAlgebra is the study of mathematical symbols and the rules for manipulating these symbols.\n\n## Key Concepts\n- Variables\n- Expressions\n- Equations"},
        {"title": "Geometry Fundamentals", "content": "# Geometry\n\nGeometry deals with shapes, sizes, and properties of space.\n\n## Topics\n- Points and Lines\n- Angles\n- Triangles"},
        {"title": "Number Systems", "content": "# Number Systems\n\n## Types of Numbers\n- Natural Numbers\n- Whole Numbers\n- Integers\n- Rational Numbers"}
    ],
    "science": [
        {"title": "The Solar System", "content": "# Our Solar System\n\nThe solar system consists of the Sun and objects that orbit it.\n\n## Planets\n1. Mercury\n2. Venus\n3. Earth\n4. Mars"},
        {"title": "States of Matter", "content": "# States of Matter\n\n## Three Main States\n- Solid\n- Liquid\n- Gas\n\n## Properties\nEach state has unique characteristics."},
        {"title": "Human Body Systems", "content": "# Body Systems\n\n## Major Systems\n- Circulatory\n- Respiratory\n- Digestive\n- Nervous"}
    ],
    "socialscience": [
        {"title": "Ancient Civilizations", "content": "# Ancient Civilizations\n\n## Key Civilizations\n- Mesopotamia\n- Egypt\n- Indus Valley\n- China"},
        {"title": "Geography Basics", "content": "# Geography\n\n## Topics\n- Maps and Globes\n- Continents\n- Oceans\n- Climate Zones"},
        {"title": "Civics Introduction", "content": "# Civics\n\n## Government\n- Democracy\n- Constitution\n- Rights and Duties"}
    ]
}

SAMPLE_MINDMAPS = {
    "math": {
        "title": "Mathematics Overview",
        "nodes": [
            {"id": "1", "label": "Mathematics", "x": 400, "y": 300},
            {"id": "2", "label": "Algebra", "x": 200, "y": 150},
            {"id": "3", "label": "Geometry", "x": 600, "y": 150},
            {"id": "4", "label": "Arithmetic", "x": 200, "y": 450},
            {"id": "5", "label": "Statistics", "x": 600, "y": 450}
        ],
        "edges": [
            {"source": "1", "target": "2"},
            {"source": "1", "target": "3"},
            {"source": "1", "target": "4"},
            {"source": "1", "target": "5"}
        ]
    },
    "science": {
        "title": "Science Branches",
        "nodes": [
            {"id": "1", "label": "Science", "x": 400, "y": 300},
            {"id": "2", "label": "Physics", "x": 200, "y": 150},
            {"id": "3", "label": "Chemistry", "x": 600, "y": 150},
            {"id": "4", "label": "Biology", "x": 200, "y": 450},
            {"id": "5", "label": "Earth Science", "x": 600, "y": 450}
        ],
        "edges": [
            {"source": "1", "target": "2"},
            {"source": "1", "target": "3"},
            {"source": "1", "target": "4"},
            {"source": "1", "target": "5"}
        ]
    },
    "socialscience": {
        "title": "Social Science Overview",
        "nodes": [
            {"id": "1", "label": "Social Science", "x": 400, "y": 300},
            {"id": "2", "label": "History", "x": 200, "y": 150},
            {"id": "3", "label": "Geography", "x": 600, "y": 150},
            {"id": "4", "label": "Civics", "x": 200, "y": 450},
            {"id": "5", "label": "Economics", "x": 600, "y": 450}
        ],
        "edges": [
            {"source": "1", "target": "2"},
            {"source": "1", "target": "3"},
            {"source": "1", "target": "4"},
            {"source": "1", "target": "5"}
        ]
    }
}

print("Content templates loaded!")
print(f"  Notes per subject: {len(SAMPLE_NOTES['math'])}")
print(f"  Mindmap per subject: 1")

## Step 9: Create School Classes

In [ ]:
print("Creating school classes...\n")

class_ids = {}  # {tenant_key: {class_div: class_id}}

# Only create classes for first 3 classes (1, 2, 3) to keep it simple
SAMPLE_CLASSES = [1, 2, 3]
SAMPLE_DIVISIONS = ["A", "B"]

for school in SCHOOLS:
    school_key = school["key"]
    tenant_id = tenant_ids.get(school_key)
    
    if not tenant_id:
        print(f"[SKIP] No tenant for {school_key}")
        continue
    
    print(f"\n--- {school['name']} ---")
    class_ids[school_key] = {}
    
    for class_num in SAMPLE_CLASSES:
        for division in SAMPLE_DIVISIONS:
            class_name = f"Class {class_num} - Division {division}"
            class_code = f"{class_num}{division}"
            
            # Check if class exists
            get_classes_url = get_url("workflow", f"/classes?tenantId={tenant_id}")
            result = api_call("GET", get_classes_url, headers={"X-Tenant-Id": tenant_id})
            
            existing_class = None
            if result["success"]:
                items = result["data"].get("items", result["data"]) if isinstance(result["data"], dict) else result["data"]
                for cls in items:
                    if cls.get("name") == class_name or cls.get("code") == class_code:
                        existing_class = cls
                        break
            
            if existing_class:
                class_ids[school_key][class_code] = existing_class["id"]
                print(f"  [EXISTS] {class_name}")
                continue
            
            # Create class
            url = get_url("workflow", "/classes")
            data = {
                "tenantId": tenant_id,
                "name": class_name,
                "code": class_code,
                "grade": class_num,
                "division": division,
                "academicYear": "2025-2026"
            }
            
            result = api_call("POST", url, data, headers={"X-Tenant-Id": tenant_id})
            
            if result["success"]:
                class_id = result["data"]["id"]
                class_ids[school_key][class_code] = class_id
                print(f"  [OK] {class_name} - ID: {class_id[:8]}...")
            else:
                print(f"  [FAIL] {class_name}: {result.get('error', '')[:50]}")

print(f"\nTotal classes created: {sum(len(c) for c in class_ids.values())}")

## Step 10: Create Sample Notes for Teachers

In [ ]:
print("Creating sample notes for teachers...\n")

note_ids = {}  # {tenant_key: [{note_id, teacher_email, class_code}]}

# Create notes for teachers in classes 1A, 1B, 2A
TEACHERS_TO_CREATE_CONTENT = [
    {"class": 1, "division": "A", "subjects": ["math", "science"]},
    {"class": 1, "division": "B", "subjects": ["math"]},
    {"class": 2, "division": "A", "subjects": ["science", "socialscience"]},
]

for school in SCHOOLS:
    school_key = school["key"]
    tenant_id = tenant_ids.get(school_key)
    
    if not tenant_id:
        continue
    
    print(f"\n--- {school['name']} ---")
    note_ids[school_key] = []
    
    for teacher_config in TEACHERS_TO_CREATE_CONTENT:
        class_num = teacher_config["class"]
        division = teacher_config["division"]
        
        for subject in teacher_config["subjects"]:
            teacher_email = f"{class_num}{division.lower()}_{subject}_t1@{school_key}.local"
            teacher_info = user_ids.get(school_key, {}).get(teacher_email)
            
            if not teacher_info:
                print(f"  [SKIP] Teacher not found: {teacher_email}")
                continue
            
            teacher_id = teacher_info["user_id"]
            print(f"\n  Teacher: {teacher_email}")
            
            # Create notes for this teacher
            for note_template in SAMPLE_NOTES.get(subject, []):
                note_title = f"{note_template['title']} - Class {class_num}{division}"
                
                url = get_url("notes", "/notes")
                data = {
                    "title": note_title,
                    "subject": SUBJECT_NAMES.get(subject, subject),
                    "grade": class_num,
                    "createdBy": teacher_id,
                    "tenantId": tenant_id,
                    "status": "DRAFT"
                }
                
                headers = {
                    "X-Tenant-Id": tenant_id,
                    "X-User-Id": teacher_id
                }
                
                result = api_call("POST", url, data, headers=headers)
                
                if result["success"]:
                    note_id = result["data"]["id"]
                    note_ids[school_key].append({
                        "note_id": note_id,
                        "teacher_email": teacher_email,
                        "teacher_id": teacher_id,
                        "class_code": f"{class_num}{division}",
                        "subject": subject,
                        "title": note_title
                    })
                    print(f"    [OK] Note: {note_title[:40]}...")
                    
                    # Add content version
                    version_url = get_url("notes", f"/notes/{note_id}/versions")
                    version_data = {
                        "content": note_template["content"],
                        "version": 1
                    }
                    api_call("POST", version_url, version_data, headers=headers)
                else:
                    print(f"    [FAIL] {note_title[:30]}: {result.get('error', '')[:40]}")

total_notes = sum(len(n) for n in note_ids.values())
print(f"\n\nTotal notes created: {total_notes}")

## Step 11: Create Sample Mindmaps for Teachers

In [ ]:
print("Creating sample mindmaps for teachers...\n")

mindmap_ids = {}  # {tenant_key: [{mindmap_id, teacher_email, class_code}]}

for school in SCHOOLS:
    school_key = school["key"]
    tenant_id = tenant_ids.get(school_key)
    
    if not tenant_id:
        continue
    
    print(f"\n--- {school['name']} ---")
    mindmap_ids[school_key] = []
    
    for teacher_config in TEACHERS_TO_CREATE_CONTENT:
        class_num = teacher_config["class"]
        division = teacher_config["division"]
        
        for subject in teacher_config["subjects"]:
            teacher_email = f"{class_num}{division.lower()}_{subject}_t1@{school_key}.local"
            teacher_info = user_ids.get(school_key, {}).get(teacher_email)
            
            if not teacher_info:
                continue
            
            teacher_id = teacher_info["user_id"]
            mindmap_template = SAMPLE_MINDMAPS.get(subject)
            
            if not mindmap_template:
                continue
            
            mindmap_title = f"{mindmap_template['title']} - Class {class_num}{division}"
            
            url = get_url("mindmap", "/mindmaps")
            data = {
                "title": mindmap_title,
                "subject": SUBJECT_NAMES.get(subject, subject),
                "grade": class_num,
                "createdBy": teacher_id,
                "tenantId": tenant_id,
                "nodes": mindmap_template["nodes"],
                "edges": mindmap_template["edges"],
                "status": "DRAFT"
            }
            
            headers = {
                "X-Tenant-Id": tenant_id,
                "X-User-Id": teacher_id
            }
            
            result = api_call("POST", url, data, headers=headers)
            
            if result["success"]:
                mindmap_id = result["data"]["id"]
                mindmap_ids[school_key].append({
                    "mindmap_id": mindmap_id,
                    "teacher_email": teacher_email,
                    "teacher_id": teacher_id,
                    "class_code": f"{class_num}{division}",
                    "subject": subject,
                    "title": mindmap_title
                })
                print(f"  [OK] Mindmap: {mindmap_title}")
            else:
                print(f"  [FAIL] {mindmap_title[:30]}: {result.get('error', '')[:40]}")

total_mindmaps = sum(len(m) for m in mindmap_ids.values())
print(f"\n\nTotal mindmaps created: {total_mindmaps}")

## Step 12: Publish and Release Content to Classes

In [ ]:
print("Publishing and releasing content to classes...\n")

# Release some notes (mark ready then release)
released_count = 0

for school_key, notes in note_ids.items():
    tenant_id = tenant_ids.get(school_key)
    if not tenant_id:
        continue
    
    print(f"\n--- {school_key} - Notes ---")
    
    # Release first 2 notes per teacher
    notes_by_teacher = {}
    for note in notes:
        teacher = note["teacher_email"]
        if teacher not in notes_by_teacher:
            notes_by_teacher[teacher] = []
        notes_by_teacher[teacher].append(note)
    
    for teacher_email, teacher_notes in notes_by_teacher.items():
        for note in teacher_notes[:2]:  # First 2 notes per teacher
            note_id = note["note_id"]
            teacher_id = note["teacher_id"]
            class_code = note["class_code"]
            class_id = class_ids.get(school_key, {}).get(class_code)
            
            headers = {
                "X-Tenant-Id": tenant_id,
                "X-User-Id": teacher_id
            }
            
            # Mark as ready
            ready_url = get_url("notes", f"/notes/{note_id}/mark-ready?readyBy={teacher_id}")
            result = api_call("POST", ready_url, headers=headers)
            
            if not result["success"]:
                print(f"  [FAIL] Mark ready: {note['title'][:30]}")
                continue
            
            # Release to class
            if class_id:
                release_url = get_url("notes", f"/notes/{note_id}/release")
                release_data = {
                    "targetClassIds": [class_id],
                    "releasedBy": teacher_id
                }
                result = api_call("POST", release_url, release_data, headers=headers)
                
                if result["success"]:
                    released_count += 1
                    print(f"  [OK] Released: {note['title'][:40]}...")
                else:
                    print(f"  [FAIL] Release: {note['title'][:30]}: {result.get('error', '')[:30]}")
            else:
                print(f"  [SKIP] No class found for {class_code}")

print(f"\n\nTotal notes released: {released_count}")

# Release mindmaps
print("\n" + "="*50)
mindmap_released = 0

for school_key, mindmaps in mindmap_ids.items():
    tenant_id = tenant_ids.get(school_key)
    if not tenant_id:
        continue
    
    print(f"\n--- {school_key} - Mindmaps ---")
    
    for mindmap in mindmaps:
        mindmap_id = mindmap["mindmap_id"]
        teacher_id = mindmap["teacher_id"]
        class_code = mindmap["class_code"]
        class_id = class_ids.get(school_key, {}).get(class_code)
        
        headers = {
            "X-Tenant-Id": tenant_id,
            "X-User-Id": teacher_id
        }
        
        # Publish mindmap
        publish_url = get_url("mindmap", f"/mindmaps/{mindmap_id}/publish")
        result = api_call("POST", publish_url, headers=headers)
        
        if not result["success"]:
            # Try PATCH to update status
            update_url = get_url("mindmap", f"/mindmaps/{mindmap_id}")
            result = api_call("PUT", update_url, {"status": "RELEASED"}, headers=headers)
        
        if result["success"]:
            mindmap_released += 1
            print(f"  [OK] Published: {mindmap['title']}")
        else:
            print(f"  [FAIL] {mindmap['title'][:30]}: {result.get('error', '')[:30]}")

print(f"\n\nTotal mindmaps published: {mindmap_released}")

## Content Summary

In [ ]:
print("="*60)
print("CONTENT CREATION SUMMARY")
print("="*60)

print(f"\nClasses created:")
for school_key, classes in class_ids.items():
    print(f"  {school_key}: {len(classes)} classes")
    for code, cid in classes.items():
        print(f"    - {code}: {cid[:8]}...")

print(f"\nNotes created:")
for school_key, notes in note_ids.items():
    print(f"  {school_key}: {len(notes)} notes")
    for note in notes[:5]:
        print(f"    - {note['title'][:40]}... (by {note['teacher_email'].split('@')[0]})")
    if len(notes) > 5:
        print(f"    ... and {len(notes) - 5} more")

print(f"\nMindmaps created:")
for school_key, mindmaps in mindmap_ids.items():
    print(f"  {school_key}: {len(mindmaps)} mindmaps")
    for mm in mindmaps:
        print(f"    - {mm['title']} (by {mm['teacher_email'].split('@')[0]})")

print("\n" + "="*60)
print("TEST SCENARIOS")
print("="*60)
print("""
1. Login as Teacher (1a_math_t1@euroschool.local / password)
   - Should see their own notes and mindmaps
   - Can create new content, publish, release to class

2. Login as Student (1a01@euroschool.local / password)
   - Should see only released content for their class (1A)
   - Cannot see drafts or other classes' content

3. Login as Principal (principal@euroschool.local / password)
   - Should see all content across all classes
   - Read-only access

4. Login as Admin (admin@euroschool.local / password)
   - Full access to all content and settings
""")